# Урок 2.1. Что такое эмбеддинги: текст числами

**Модуль 2 «Embeddings и векторные базы данных» · Курс «LLM для Python-разработчиков»**

Добро пожаловать во второй модуль! В первом вы научились *разговаривать* с моделью. Теперь научимся *искать по смыслу* — это вторая опора всех LLM-систем и фундамент RAG, к которому мы идём.

## 🎯 Цели урока

После этого урока вы сможете:

- объяснить идею **эмбеддинга** (embedding): текст → вектор чисел, близкий смысл → близкие векторы;
- считать **косинусную близость** (cosine similarity) руками и понимать, почему угол, а не расстояние;
- рассказать, откуда в векторах берётся «смысл» — **дистрибутивная гипотеза**, которую вы проверите экспериментом;
- пользоваться **sentence-transformers** и знать критичный нюанс модели e5 — префиксы `query:` / `passage:`;
- собрать свой первый **семантический поиск** по базе знаний компании.

## План

1. Задача, которую не решить промптом
2. Наивные векторы: bag of words
3. Косинусная близость
4. Дистрибутивная гипотеза — эксперимент
5. Настоящие эмбеддинги: sentence-transformers
6. Семантический поиск по базе знаний «Векторики»

⏱ **Время:** ~75–90 минут.

> 💡 Первая половина урока считается на чистом numpy и работает сразу. Для второй понадобится скачать модель эмбеддингов (~120 МБ, один раз) — ячейки подскажут как; до этого они вежливо пропускаются.

## 1. Задача, которую не решить промптом

Финал модуля 1 закончился классификатором тикетов: четыре категории, описанные в промпте. А теперь типичная задача из жизни: **«найди среди 10 000 документов компании те, что отвечают на вопрос пользователя»**. В промпт их не положить — контекст не резиновый (урок 1.1). Few-shot не растянуть на тысячи примеров (урок 1.7).

Классический поиск по словам (`LIKE '%...%'`, полнотекстовые индексы) знаком каждому бэкендеру — и все знают, где он ломается:

- пользователь пишет «**отгул**», в документе — «**отпуск без сохранения**»: ноль совпадений;
- «**не работает интернет**» и «**проблемы с сетью**» — об одном и том же, но не совпадают ни словом;
- «**ключ**» находится и в «API-ключ», и в «ключ от офиса» — совпадение слов ≠ совпадение смысла.

Нужно сравнивать не буквы, а **смысл**. Но компьютер умеет сравнивать только числа. Значит, задача сводится к одному: **превратить текст в такие числа, чтобы у близких по смыслу текстов числа тоже были близкими.**

Такое представление называется **эмбеддингом** (embedding — «вложение» текста в числовое пространство). Полезная аналогия: это как хеш-функция наоборот. Хеш *специально разрушает* близость — `md5("отпуск")` и `md5("отпускк")` не имеют ничего общего. Эмбеддинг — «хеш, **сохраняющий** смысл»: близкие тексты → близкие векторы. Или как координаты на карте: широта и долгота ничего не говорят сами по себе, но по ним можно посчитать, что рядом, а что далеко. Эмбеддинг — координаты текста на «карте смыслов» в сотнях измерений.

Идею проще всего понять, построив плохой эмбеддинг своими руками — и посмотрев, где он сломается.

## 2. Наивные векторы: считаем слова

Первое, что приходит в голову: составить словарь всех слов и представить текст **вектором счётчиков** — сколько раз каждое слово встретилось. Подход называется **bag of words** («мешок слов» — порядок выбрасываем, храним только счётчики):

In [ ]:
import re

import numpy as np

PHRASES = [
    "кот ловит мышей",
    "кот спит на диване",
    "собака охраняет дом",
    "пёс лает на прохожих",
    "сервер обрабатывает запросы",
    "база данных хранит записи",
]

def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())

vocab = sorted({word for phrase in PHRASES for word in tokenize(phrase)})
print(f"Словарь ({len(vocab)} слов): {vocab}\n")

def bow_vector(text: str) -> np.ndarray:
    """Вектор счётчиков слов: одно измерение на слово словаря."""
    counts = np.zeros(len(vocab))
    for word in tokenize(text):
        if word in vocab:
            counts[vocab.index(word)] += 1
    return counts

for phrase in PHRASES[:2]:
    print(f"{phrase!r}")
    print(f"  -> {bow_vector(phrase).astype(int)}")

Каждая фраза стала точкой в 19-мерном пространстве. Теперь нужен способ измерять, насколько две точки «про одно и то же».

## 3. Косинусная близость: сравниваем направление, а не длину

Первое желание — евклидово расстояние между точками. Но у него есть проблема: **длина вектора растёт с длиной текста**. Напишите ту же мысль в два раза длиннее — счётчики удвоятся, вектор «уедет» вдвое дальше от начала координат, хотя смысл не изменился.

Поэтому сравнивают не позиции точек, а **направления векторов** — угол между ними:

```text
        ^
        |      A («кот ловит мышей»)
        |     /
        |    /         угол маленький -> тексты похожи
        |   /   B («кот спит на диване»)
        |  /  _/
        | / _/
        |/_/________________>
       0
        \_
          \__  C («сервер обрабатывает запросы»)
             \_,   угол с A почти прямой -> общего мало
```

Метрика — косинус этого угла:

```text
cosine(A, B) = (A · B) / (|A| × |B|)
```

Числитель — скалярное произведение (совпадающие измерения), знаменатель — нормировка на длины: длинный и короткий текст сравниваются честно. Диапазон: от −1 до 1; для счётчиков слов (неотрицательных) — от 0 (ничего общего) до 1 (одно направление).

In [ ]:
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """Косинус угла между векторами."""
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / norm) if norm else 0.0

# матрица близости всех фраз со всеми
print(" " * 32 + "  ".join(f"№{j + 1}  " for j in range(len(PHRASES))))
for i, phrase_a in enumerate(PHRASES):
    row = "  ".join(
        f"{cosine(bow_vector(phrase_a), bow_vector(phrase_b)):.2f}"
        for phrase_b in PHRASES
    )
    print(f"№{i + 1} {phrase_a:<28} {row}")

print("\nКлючевые пары:")
print(f"  «кот ловит мышей» <-> «кот спит на диване»:      "
      f"{cosine(bow_vector(PHRASES[0]), bow_vector(PHRASES[1])):.2f}")
print(f"  «собака охраняет дом» <-> «пёс лает на прохожих»: "
      f"{cosine(bow_vector(PHRASES[2]), bow_vector(PHRASES[3])):.2f}")

Механика работает: две фразы про кота получили близость 0.29 (общее слово), фразы про сервер и кота — ноль. Но посмотрите на вторую ключевую пару: **«собака охраняет дом» и «пёс лает на прохожих» — 0.00**. Для человека это соседние предложения из одного рассказа; для bag of words «собака» и «пёс» — просто разные измерения вектора, между ними нет ничего общего, как между «сервер» и «диване».

Это фундаментальный потолок подсчёта слов: **синонимы ортогональны**. Никакие счётчики не скажут машине, что пёс — это собака.

А что скажет? В 1957 году лингвист Джон Фёрс сформулировал **дистрибутивную гипотезу** — идею, на которой стоят все современные эмбеддинги: *смысл слова определяется контекстами, в которых оно встречается*. «Пёс» и «собака» встречаются рядом с одними и теми же словами — лает, охраняет, хвост, поводок. Если описать слово через его соседей — у синонимов описания совпадут **сами собой**.

## 4. Дистрибутивная гипотеза: проверяем экспериментом

Звучит красиво, но давайте проверим руками — безо всяких нейросетей. Возьмём мини-корпус предложений и опишем каждое слово вектором: **с какими словами оно встречается в окне ±2**:

In [ ]:
CORPUS = [
    "собака громко лает во дворе",
    "пёс громко лает во дворе",
    "собака охраняет наш дом",
    "пёс охраняет наш дом ночью",
    "наша собака любит гулять в парке",
    "наш пёс любит гулять в парке",
    "собака виляет хвостом от радости",
    "пёс виляет хвостом от радости",
    "кот спит на тёплом диване",
    "кот ловит мышей в подвале",
    "наш кот любит гулять сам по себе",
    "кот мурлычет от радости",
    "сервер обрабатывает сетевые запросы",
    "сервер упал под высокой нагрузкой",
    "база данных хранит записи о заказах",
    "база данных обрабатывает сложные запросы",
]
WINDOW = 2  # сколько соседей слева и справа считаем «контекстом»

cooc_vocab = sorted({word for sentence in CORPUS for word in sentence.split()})
word_index = {word: i for i, word in enumerate(cooc_vocab)}
cooc = np.zeros((len(cooc_vocab), len(cooc_vocab)))

for sentence in CORPUS:
    words = sentence.split()
    for i, word in enumerate(words):
        for j in range(max(0, i - WINDOW), min(len(words), i + WINDOW + 1)):
            if i != j:
                cooc[word_index[word], word_index[words[j]]] += 1

def word_vector(word: str) -> np.ndarray:
    """Вектор слова = счётчики его соседей по корпусу."""
    return cooc[word_index[word]]

print(f"Словарь: {len(cooc_vocab)} слов, вектор каждого слова — {len(cooc_vocab)} измерений\n")
for other in ("собака", "кот", "сервер"):
    print(f"пёс <-> {other:<7} {cosine(word_vector('пёс'), word_vector(other)):.3f}")

Смотрите внимательно: слова «пёс» и «собака» **ни разу не встретились вместе** — но их векторы почти совпали, потому что совпали их *соседи*. Кот оказался на среднем расстоянии (он тоже «наш», тоже «любит гулять»), сервер — далеко. Смысловая близость появилась из голой статистики соседства. Ни одной нейросети.

Современные нейросетевые эмбеддинги — это та же идея, доведённая до предела:

- вместо мини-корпуса — **триллионы токенов** (как pretraining из урока 1.1 — и это не совпадение: трансформер-эмбеддер — близкий родственник LLM);
- вместо счётчиков соседей — **плотный вектор** фиксированной длины (у нашей модели — 384 измерения), в который сеть *сжимает* контекстную статистику;
- вместо отдельных слов — **целые фразы и абзацы**, с учётом порядка слов и многозначности: «ключ» в «API-ключ» и «ключ от офиса» получит *разные* векторы, потому что модель смотрит на контекст (attention из урока 1.1 наконец-то работает на нас).

## 5. Настоящие эмбеддинги: sentence-transformers

Рабочая лошадка модуля — библиотека **sentence-transformers** и модель **`intfloat/multilingual-e5-small`**: 384 измерения, ~120 МБ, понимает 100+ языков включая русский, комфортно работает даже на CPU. Ставим и скачиваем (модель приедет с huggingface.co при первом использовании и осядет в локальном кэше):

In [ ]:
# Один раз: установка библиотеки (потянет за собой torch, это нормально)
%pip install -q sentence-transformers

In [ ]:
EMB_MODEL_NAME = "intfloat/multilingual-e5-small"

try:
    from sentence_transformers import SentenceTransformer
    ST_INSTALLED = True
except ImportError:
    ST_INSTALLED = False

model = None
if ST_INSTALLED:
    try:
        # local_files_only: берём модель ТОЛЬКО из кэша, без похода в сеть
        from huggingface_hub import snapshot_download
        snapshot_download(EMB_MODEL_NAME, local_files_only=True)
        model = SentenceTransformer(EMB_MODEL_NAME)
        print(f"Модель {EMB_MODEL_NAME} загружена из локального кэша.")
    except Exception:
        print(f"Модель {EMB_MODEL_NAME} ещё не скачана (~120 МБ).")
        print("Выполните следующую ячейку — она скачает модель один раз.")
else:
    print("Пакет sentence-transformers не установлен — выполните ячейку с %pip install выше.")

EMB_AVAILABLE = model is not None

In [ ]:
# Скачивание модели (~120 МБ) — нужно выполнить ОДИН раз при живом интернете.
# Дальше все запуски читают её из кэша (~/.cache/huggingface).
if ST_INSTALLED and not EMB_AVAILABLE:
    model = SentenceTransformer(EMB_MODEL_NAME)
    EMB_AVAILABLE = True
    print("Готово, модель в кэше.")
elif EMB_AVAILABLE:
    print("Модель уже в кэше — скачивать нечего.")

In [ ]:
def embed_texts(texts: list[str], *, prefix: str) -> np.ndarray | None:
    """Эмбеддинги списка текстов.

    prefix — обязательная особенность моделей семейства e5 (см. ниже):
    "query" для запросов и сравнений, "passage" для индексируемых документов.
    normalize_embeddings=True приводит векторы к длине 1 — тогда
    косинус считается простым скалярным произведением (a @ b).
    """
    if not EMB_AVAILABLE:
        print("[пропущено: модель не загружена — вернитесь после скачивания]")
        return None
    return model.encode(
        [f"{prefix}: {text}" for text in texts],
        normalize_embeddings=True,
    )

vectors = embed_texts(
    ["Кот спит на диване.", "Пёс лает во дворе.", "Сервер упал под нагрузкой."],
    prefix="query",
)
if vectors is not None:
    print("Форма матрицы:", vectors.shape)  # 3 текста x 384 измерения
    print("Первые 6 чисел первого вектора:", np.round(vectors[0][:6], 3))

Вы получите матрицу **(3, 384)**: три текста, по 384 числа на каждый. Сами числа не значат ничего (это координаты на «карте смыслов», а карта у каждой модели своя) — смысл появляется только **в сравнении** векторов между собой.

Устроим нейросети тот же экзамен, на котором засыпался bag of words, — синонимы и перефразы:

In [ ]:
TEST_PAIRS = [
    ("собака охраняет дом",   "пёс лает во дворе"),        # bag of words дал 0.00!
    ("не работает интернет",  "проблемы с сетью"),          # ни одного общего слова
    ("не работает интернет",  "рецепт борща с пампушками"), # контроль: разные темы
]

if EMB_AVAILABLE:
    for text_a, text_b in TEST_PAIRS:
        vec_a, vec_b = embed_texts([text_a, text_b], prefix="query")
        print(f"{text_a!r} <-> {text_b!r}: {float(vec_a @ vec_b):.3f}")
else:
    print("[пропущено: модель не загружена — вернитесь после скачивания]")

Ожидаемая картина (точные числа у каждой модели свои): первые две пары получат **высокую** близость, контрольная пара про борщ — **заметно ниже**. Собака и пёс наконец-то встретились: модель выучила дистрибутивную гипотезу на триллионах токенов — так же, как наш эксперимент на 16 предложениях, только по-настоящему.

> ⚠️ **Нюанс, который все пропускают: префиксы e5.** Модели семейства e5 обучались с служебными префиксами в тексте, и без них качество **тихо деградирует** — ошибок не будет, просто поиск станет хуже. Правило:
>
> | задача | текст A | текст B |
> |---|---|---|
> | сравнить два текста | `query: ...` | `query: ...` |
> | поиск: запрос → документы | `query: ...` | `passage: ...` |
>
> Наш `embed_texts` делает это за вас. У других моделей свои конвенции (у многих префиксов нет вовсе) — **читайте карточку модели**. Это буквально первый вопрос при любых проблемах с качеством поиска.

## 6. Семантический поиск по базе знаний «Векторики»

Пора на настоящие данные. В папке `data/` этого модуля лежит база знаний вымышленной IT-компании «Векторика» — 14 документов: HR-политики, IT-инструкции, регламенты разработки. Это наш сквозной пример до конца курса: в модуле 3 мы построим по нему полноценный RAG.

Рядом лежит [`utils.py`](utils.py) с готовым загрузчиком (внутри — обычные dataclass'ы, загляните):

In [ ]:
from utils import load_documents

docs = load_documents("data")
print(f"Документов: {len(docs)}\n")
for doc in docs:
    print(f"{doc.doc_id:<15} [{doc.topic:<6}] {doc.title}")

In [ ]:
# Индексация: каждый документ -> вектор (префикс "passage" - мы ИНДЕКСИРУЕМ).
# Поиск: вопрос -> вектор (префикс "query") -> косинус со всеми документами.
if EMB_AVAILABLE:
    doc_vectors = embed_texts([doc.text for doc in docs], prefix="passage")

    for question in ("Как оформить отгул на пару дней?",
                     "Что делать, если упал прод?"):
        query_vec = embed_texts([question], prefix="query")[0]
        scores = doc_vectors @ query_vec          # косинусы со всеми документами
        top = np.argsort(scores)[::-1][:3]        # три ближайших
        print(f"Вопрос: {question}")
        for idx in top:
            print(f"  {scores[idx]:.3f}  {docs[idx].doc_id:<14} {docs[idx].title}")
        print()
else:
    print("[пропущено: модель не загружена — вернитесь после скачивания]")

Обратите внимание на первый вопрос: слова «отгул» **нет ни в одном документе** «Векторики». Полнотекстовый поиск вернул бы пустоту; семантический — приносит документы про отпуск и больничный, потому что они *рядом на карте смыслов*. Это и есть momento, ради которого существует весь модуль.

Что здесь пока плохо — и что мы будем чинить:

1. **Полный перебор.** Мы посчитали косинус запроса с *каждым* документом: O(N) на запрос. На 14 документах — мгновенно, на миллионе — сотни миллисекунд и гигабайты векторов в памяти. Решение — специальные индексы и **векторные базы данных**: следующий урок.
2. **Вектор на целый документ.** Документ о больничном говорит и про оформление, и про оплату, и про COVID-правила — его вектор оказывается «средним по больнице», и длинные документы размывают поиск. Решение — **чанкинг**: уроки 2.5–2.6.
3. **Поиск на глазок.** Мы оценили результаты взглядом; в проде нужен eval с метриками — как в уроке 1.7, и он появится в упражнении 2.

## ⚠️ Частые ошибки

1. **Путать сходство и расстояние.** В одних библиотеках больше = ближе (cosine similarity), в других меньше = ближе (cosine distance = 1 − similarity). Проверяйте, что возвращает ваша функция, прежде чем сортировать.
2. **Игнорировать конвенции модели.** Префиксы `query:`/`passage:` у e5 — не украшение: без них качество тихо проседает. У каждой модели своя инструкция — читайте карточку.
3. **Эмбеддить длинные тексты целиком.** Вектор «обо всём сразу» не близок ни к чему конкретному. Длинное — резать (чанкинг, уроки 2.5–2.6).
4. **Ждать от эмбеддингов точного совпадения терминов.** Номер заказа, код ошибки, артикул — территория keyword-поиска. Смысловой и точный поиск дополняют друг друга (гибрид — впереди).
5. **Смешивать модели.** Проиндексировали одной моделью, ищете другой — векторы из разных «карт», сравнивать их бессмысленно (а часто и размерности не совпадут). Сменили модель — переиндексируйте всё.
6. **Забыть про язык.** Моноязычная английская модель на русских текстах даст мусор без единой ошибки в логах. Наша e5 — multilingual, и это осознанный выбор.

## 🏋️ Упражнения

### Упражнение 1 — карта многозначности 🗝

Возьмите 6 фраз, где одинаковые слова означают разное и разные слова — одно и то же (классика: «ключ от квартиры» / «API-ключ» / «ключ к успеху»). Постройте матрицу близостей и для каждой фразы найдите ближайшую. Убедитесь, что модель разводит смыслы одного слова по контексту — то, что bag of words не смог бы в принципе.

### Упражнение 2 — eval для поиска 📊

Как в уроке 1.7: замер вместо взгляда. Составьте 5 вопросов к базе «Векторики» с ожидаемым `doc_id`, посчитайте **top-1 accuracy** (правильный документ первый) и **top-3 hit rate** (правильный в тройке). Этот мини-eval поедет с нами в следующие уроки — сравнивать стратегии поиска.

### Упражнение 3 — хомяк в векторном пространстве 🐹

Без нейросети: добавьте в корпус из раздела 4 пять предложений про хомяка так, чтобы его вектор оказался ближе к коту и псу, чем к серверу. Пересчитайте матрицу соседства и проверьте. Получилось с первого раза? Что решает — само слово «хомяк» или соседи, которых вы ему выбрали?

## 📝 Мини-квиз

**1. Чем эмбеддинг принципиально отличается от хеша?**

<details><summary>Ответ</summary>

Хеш специально *разрушает* близость: похожие входы дают несвязанные выходы. Эмбеддинг — наоборот, *сохраняет*: близкие по смыслу тексты получают близкие векторы. Именно это свойство позволяет искать «похожее», а не только «точно такое же».

</details>

**2. Почему тексты сравнивают косинусом, а не евклидовым расстоянием?**

<details><summary>Ответ</summary>

Длина вектора зависит от длины текста (у счётчиков — напрямую), а смысл — нет. Косинус сравнивает *направления*, нормируя на длины, поэтому короткая и длинная формулировки одной мысли остаются близкими. (С нормированными векторами обе метрики, впрочем, эквивалентны по порядку сортировки.)

</details>

**3. Сформулируйте дистрибутивную гипотезу и объясните, причём тут «пёс» и «собака».**

<details><summary>Ответ</summary>

«Смысл слова определяется контекстами, в которых оно встречается». «Пёс» и «собака» почти не встречаются *вместе*, но встречаются *рядом с одними и теми же словами* — поэтому векторы, построенные из статистики соседей, у них совпадают. Мы воспроизвели это на 16 предложениях без всяких нейросетей.

</details>

**4. Зачем модели e5 префиксы `query:` и `passage:` и что будет без них?**

<details><summary>Ответ</summary>

Модель обучалась с этими префиксами: они говорят ей, «смотреть» на текст как на запрос или как на документ. Без префиксов ошибки не будет — просто качество поиска тихо просядет. Худший вид бага: невидимый.

</details>

**5. Команда переехала на новую модель эмбеддингов, но старые векторы в базе оставила «чтобы не переиндексировать». Что будет?**

<details><summary>Ответ</summary>

Поиск превратится в лотерею: векторы разных моделей живут в разных «системах координат», их косинусы бессмысленны (если размерности вообще совпадут). Правило жёсткое: одна коллекция — одна модель; сменили модель — переиндексировали всё.

</details>

## ✅ Решения упражнений

### Решение 1 — карта многозначности

In [ ]:
KEY_PHRASES = [
    "потерял ключ от квартиры",
    "сгенерируйте новый API-ключ для сервиса",
    "ключ к успеху — дисциплина",
    "дверь захлопнулась, не могу попасть домой",
    "токен авторизации перестал работать",
    "как достичь цели и не сдаваться",
]

if EMB_AVAILABLE:
    vectors = embed_texts(KEY_PHRASES, prefix="query")
    sims = vectors @ vectors.T
    np.fill_diagonal(sims, -1.0)          # чтобы фраза не находила саму себя
    for i, phrase in enumerate(KEY_PHRASES):
        j = int(np.argmax(sims[i]))
        print(f"{phrase!r}")
        print(f"   ближайшая: {KEY_PHRASES[j]!r}  ({sims[i, j]:.3f})\n")
else:
    print("[пропущено: модель не загружена — вернитесь после скачивания]")

Ожидаемые пары: квартирный ключ — к захлопнувшейся двери, API-ключ — к токену, «ключ к успеху» — к целям. Слово «ключ» встречается в трёх фразах, но модель разводит их по разным районам «карты» — потому что вектор строится из всего контекста, а не из отдельных слов.

### Решение 2 — eval для поиска

In [ ]:
EVAL_QUESTIONS = [
    ("Сколько дней отпуска положено в году?",              "otpusk"),
    ("Как оформить больничный лист?",                      "bolnichny"),
    ("VPN подключён, но внутренние сайты не открываются",  "vpn"),
    ("Кто должен посмотреть мой код перед мержем?",        "code-review"),
    ("Упал прод, что делать в первую очередь?",            "incidents"),
]

if EMB_AVAILABLE:
    doc_vectors = embed_texts([doc.text for doc in docs], prefix="passage")
    top1 = top3 = 0
    for question, expected in EVAL_QUESTIONS:
        query_vec = embed_texts([question], prefix="query")[0]
        ranked = np.argsort(doc_vectors @ query_vec)[::-1]
        top_ids = [docs[i].doc_id for i in ranked[:3]]
        top1 += top_ids[0] == expected
        top3 += expected in top_ids
        status = "+" if top_ids[0] == expected else ("~" if expected in top_ids else "x")
        print(f"[{status}] {question}")
        print(f"     ждали {expected!r}, топ-3: {top_ids}")
    total = len(EVAL_QUESTIONS)
    print(f"\ntop-1 accuracy: {top1}/{total}   top-3 hit rate: {top3}/{total}")
else:
    print("[пропущено: модель не загружена — вернитесь после скачивания]")

Числа важнее впечатлений: когда в уроке 2.5 мы начнём резать документы на чанки, именно этот eval покажет, стало лучше или хуже. Сохраните свои вопросы — добавим к ним новые.

### Решение 3 — хомяк в векторном пространстве

In [ ]:
HAMSTER_SENTENCES = [
    "хомяк спит в уютной клетке",
    "наш хомяк любит гулять в колесе",
    "хомяк грызёт зерно от радости",
    "наш хомяк любит гулять в парке",
    "хомяк виляет хвостом редко",
]

corpus_v2 = CORPUS + HAMSTER_SENTENCES

vocab_v2 = sorted({word for sentence in corpus_v2 for word in sentence.split()})
index_v2 = {word: i for i, word in enumerate(vocab_v2)}
cooc_v2 = np.zeros((len(vocab_v2), len(vocab_v2)))
for sentence in corpus_v2:
    words = sentence.split()
    for i, word in enumerate(words):
        for j in range(max(0, i - WINDOW), min(len(words), i + WINDOW + 1)):
            if i != j:
                cooc_v2[index_v2[word], index_v2[words[j]]] += 1

for other in ("кот", "пёс", "собака", "сервер"):
    sim = cosine(cooc_v2[index_v2["хомяк"]], cooc_v2[index_v2[other]])
    print(f"хомяк <-> {other:<7} {sim:.3f}")

Хомяк встал рядом с животными — не потому, что компьютер «знает» зоологию, а потому, что мы дали ему те же контексты: «наш … любит гулять», «спит», «от радости». Уберите эти совпадения — и хомяк уплывёт в сторону серверов. Смысл в эмбеддингах — это контексты, и ничего кроме контекстов. Теперь вы понимаете эту фразу буквально.

## 🎓 Итоги

- **Эмбеддинг** — «хеш, сохраняющий смысл»: текст → вектор, близкий смысл → близкие векторы. Сравниваем **косинусом** — он смотрит на направление, а не длину.
- Bag of words ломается на синонимах: «пёс» и «собака» ортогональны. Лечит **дистрибутивная гипотеза** — смысл слова определяется его соседями; вы проверили её экспериментом на numpy.
- Нейросетевые эмбеддинги — та же идея на триллионах токенов: 384 плотных измерения, целые фразы, многозначность через контекст.
- **sentence-transformers + multilingual-e5-small** — наш рабочий стек; префиксы `query:`/`passage:` обязательны, карточка модели — закон.
- Семантический поиск нашёл «отгул» в документах, где этого слова нет. Но пока это полный перебор O(N) с вектором на целый документ — обе проблемы решаем дальше.

**Дальше — урок 2.2 «Векторные базы данных»:** как искать среди миллионов векторов за миллисекунды (ANN, HNSW на пальцах) и чем отличаются Chroma, Qdrant, pgvector и FAISS.

## 📚 Что почитать

- [Документация sentence-transformers](https://sbert.net/) — библиотека, с которой мы работаем
- [Карточка intfloat/multilingual-e5-small](https://huggingface.co/intfloat/multilingual-e5-small) — те самые префиксы и детали обучения
- [Jurafsky & Martin, Speech and Language Processing, гл. 6](https://web.stanford.edu/~jurafsky/slp3/6.pdf) — векторная семантика: от счётчиков до эмбеддингов (лучшая глава по теме)
- [Illustrated Word2vec](https://jalammar.github.io/illustrated-word2vec/) — визуальное объяснение первых нейроэмбеддингов
- [MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard) — рейтинг моделей эмбеддингов: пригодится, когда захотите выбрать свою